![TensorFlow](https://upload.wikimedia.org/wikipedia/commons/thumb/a/ab/TensorFlow_logo.svg/960px-TensorFlow_logo.svg.png)


In [ ]:
import tensorflow as tf
import keras as K
import numpy as np

In [ ]:
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {K.__version__}")

# Tensor Manipulation Basics

### Tensor Creation

A 0-dimensional tensor is a __scalar__

In [ ]:
scalar = tf.constant(0)
print(f"Value of the tensor = {scalar}")
print(f"Number of dimensions = {len(scalar.shape)}")
print(f"Tensor's shape = {scalar.shape}")

A 1-dimensional tensor is a __vector__

In [ ]:
vector = tf.constant([1, 2, 3])
print(f"Value of the tensor = {vector}")
print(f"Number of dimensions = {len(vector.shape)}")
print(f"Tensor's shape = {vector.shape}")

A 2-dimensional tensor is a __matrix__

In [ ]:
matrix = tf.constant(
    [[1, 2, 3],
     [4, 5, 6],
     [7, 8, 9]]
)
print(f"Value of the tensor = \n {matrix}")
print(f"Number of dimensions = {len(matrix.shape)}")
print(f"Tensor's shape = {matrix.shape}")

We can generalize tensors to __n dimensions__

In [ ]:
n = 3
tensor = tf.random.normal(tuple(3 for _ in range(n)))
print(f"Value of the tensor = \n {tensor}")
print(f"Number of dimensions = {len(tensor.shape)}")
print(f"Tensor's shape = {tensor.shape}")

Each tensor is characterized also by a __data type__

In [ ]:
tensor.dtype

Which can be cast to others (with the clear consequences on the numerical representation)

In [ ]:
int_tensor = tf.cast(tensor, dtype=tf.int32)
print(int_tensor)

### Tensor indexing and slicing

Tensors can be __indexed__ (i.e., ``tensor[i, :, :]``) or __sliced__ (i.e., ``tensor[:i, ...]``).

Indexing a tensor reduces its dimensionality depending to the number of "free" dimensions.

In [ ]:
print(f"Scalar = {tensor[0, 1, 0]}")
print(f"Vector = {tensor[:, 0, -1]}") # : means all elements in that dimension
print(f"Matrix = {tensor[:, 2]}")

Slicing reduces the size of the sliced dimension. With this approach, only contiguous slices can be taken. To get scattered slices, use [``tf.gather``](https://www.tensorflow.org/api_docs/python/tf/gather).

In [ ]:
tensor_slice = tensor[1:]
print(f"tensor_slice = {tensor_slice}")
print(f"Shape of tensor_slice = {tensor_slice.shape}")
print(f"Number of dimensions = {len(tensor_slice.shape)}")

In [ ]:
tensor_slice = tensor[1:, :, -1:]
print(f"tensor_slice = {tensor_slice}")
print(f"Shape of tensor_slice = {tensor_slice.shape}")
print(f"Number of dimensions = {len(tensor_slice.shape)}")

Indexing and slicing can be also mixed.

In [ ]:
tensor_slice = tensor[1:, :, 2]
print(f"tensor_slice = {tensor_slice}")
print(f"Shape of tensor_slice = {tensor_slice.shape}")
print(f"Number of dimensions = {len(tensor_slice.shape)}")

### Concat and stack

- ``tf.concat``: concatenates n tensors on the `axis` dimension. All the dimensions of the input tensors, except for the `axis` dimension, must match.
- ``tf.stack``: stacks n tensors on the `axis` dimension, which is added for the resulting tensor. All the dimensions of the input tensors must match.

In [ ]:
tensor_1 = tf.random.normal((2, 3, 4))
tensor_2 = tf.random.uniform((2, 6, 4))
concat_tensor = tf.concat([tensor_1, tensor_2], axis=1)
print(f"Shape = {concat_tensor.shape}")

In [ ]:
tensor_2 = tf.random.uniform((2, 3, 4))
stacked_tensor = tf.stack([tensor_1, tensor_2], axis=1)
print(f"Shape = {stacked_tensor.shape}")

### Coming from PyTorch

TensorFlow and PyTorch share the same NumPy-inspired semantics for tensor manipulation, but there are a few differences worth noting.

#### Mutability

In PyTorch every tensor is **mutable** — you can modify it in-place with operators that have a trailing `_` (e.g. `add_`, `mul_`). In TensorFlow, `tf.constant` produces an **immutable** tensor. If you need mutable state (e.g. model parameters), you must use `tf.Variable`.

In [ ]:
import torch

# PyTorch: tensors are mutable
pt = torch.tensor([1.0, 2.0, 3.0])
pt.add_(10)          # in-place: pt is now [11, 12, 13]
pt[0] = 0.0          # item assignment works
print("PyTorch after in-place ops:", pt)

In [ ]:
# TensorFlow: tf.constant is immutable
tc = tf.constant([1.0, 2.0, 3.0])
# tc[0].assign(0.0)  # AttributeError — constants have no .assign()
# There is no add_() either; every operation returns a *new* tensor
tc2 = tc + 10
print("Original constant:", tc)
print("New tensor after +10:", tc2)

In [ ]:
try:
    tc[0] = 0.0
except TypeError as e:
    print(e)

In [ ]:
# TensorFlow: tf.Variable is mutable
tv = tf.Variable([1.0, 2.0, 3.0])
tv.assign_add([10, 10, 10])   # in-place add
tv[0].assign(0.0)             # item assignment works
print("Variable after in-place ops:", tv)

#### Type strictness

PyTorch **auto-promotes** dtypes: mixing `int` and `float` tensors silently produces a `float` result. TensorFlow is **strict** — operands must have matching dtypes or you get an error. You must cast explicitly with `tf.cast`.

In [ ]:
# PyTorch: implicit promotion
a_pt = torch.tensor([1, 2, 3])          # int64
b_pt = torch.tensor([0.1, 0.2, 0.3])    # float32
print("PyTorch int + float =", a_pt + b_pt)  # works, result is float32

In [ ]:
# TensorFlow: strict dtype matching
a_tf = tf.constant([1, 2, 3])            # int32
b_tf = tf.constant([0.1, 0.2, 0.3])      # float32
try:
    a_tf + b_tf  # TypeError!
except (TypeError, tf.errors.InvalidArgumentError) as e:
    print(f"TF error: {e}")

# Fix: explicit cast
print("After tf.cast:", tf.cast(a_tf, tf.float32) + b_tf)

#### Reshaping and memory layout

In PyTorch, `view` and `reshape` behave differently:
- `view` **requires contiguous memory** — it fails on tensors whose memory layout was broken by `transpose` or `permute`;
- `reshape` **always works** — it returns a view when possible, or silently copies the data otherwise.

TensorFlow has **only `tf.reshape`**, and it behaves like PyTorch's `reshape`: it always succeeds. However, there is an important difference: since TF tensors are immutable, the framework is free to decide internally whether to share or copy the underlying buffer — the user cannot observe the difference. There is no `view` equivalent and no notion of contiguity exposed to the user.

In [ ]:
# PyTorch: view vs reshape after transpose
x_pt = torch.arange(6).reshape(2, 3)
print("Original:\n", x_pt)

xt_pt = x_pt.t()  # transpose — makes the tensor non-contiguous
print(f"\nTransposed (contiguous? {xt_pt.is_contiguous()}):\n", xt_pt)

try:
    xt_pt.view(6)  # fails: non-contiguous!
except RuntimeError as e:
    print(f"\nview error: {e}")

print("reshape works:", xt_pt.reshape(6))           # silently copies
print("contiguous().view works:", xt_pt.contiguous().view(6))  # explicit copy then view

In [ ]:
# TensorFlow: tf.reshape always succeeds — no contiguity concept
x_tf = tf.constant([[0, 1, 2], [3, 4, 5]])
print("Original:\n", x_tf)

xt_tf = tf.transpose(x_tf)
print("\nTransposed:\n", xt_tf)

flat_tf = tf.reshape(xt_tf, [-1])  # always works
print("\nReshaped (flat):", flat_tf)

Notice the **element order** in the flattened tensors above: both PyTorch's `reshape` and TensorFlow's `tf.reshape` read elements in **row-major (C) order from the current logical layout**. After a transpose the logical layout changes, so flattening a transposed tensor gives `[0, 3, 1, 4, 2, 5]` (columns of the original), not `[0, 1, 2, 3, 4, 5]`.

---
# Keras Basics

Keras is a high-level library leveraging TensorFlow as backend. It allows to build and experiment with ML models with an easy and flexible (?) interface.

API Docs (go to `tf.keras`): https://www.tensorflow.org/versions

Keras Guides (for specific API components): https://www.tensorflow.org/guide/keras

Tutorials (cover examples of basic usage): https://www.tensorflow.org/tutorials/keras

**Warning:** in general, it is better to use Keras whenever it is possible, since it exploits compiled computational graphs, making it way more efficent than TensorFlow (with eager execution activated, which is the default behaviour).

In [ ]:
import seaborn as sns
import pandas as pd
sns.set_theme()

In [ ]:
from sklearn.model_selection import train_test_split

(train_X, train_y), (test_X, test_y) = K.datasets.mnist.load_data(path='ds')
train_X, eval_X, train_y, eval_y = train_test_split(
    train_X, train_y,
    test_size=0.15,
    shuffle=True,
    stratify=train_y
)

# Model Interfaces in Keras

Now we will see how to solve the same problem we've seen before with these three interfaces.

## Sequential API

Everything is wrapped within the ``K.Sequential()`` "box", and all the objects that you add to this box are considered layers. Note that, here, a layer is not necessarily a neural layer, but whatever applies a transformation to the output of the previous layer (or to the input data).

In general, you can consider the K.Sequential() interface as a wrapper for the pipeline which applies all the subsequent transformations to your input data, ending to the prediction of your model.

In [ ]:
seq_model = K.Sequential()

seq_model.add(K.Input(shape=(28, 28))) # What happens if I remove the input?
seq_model.add(K.layers.Flatten())
seq_model.add(K.layers.Rescaling(scale=1./255))
seq_model.add(K.layers.Dense(10, activation='softmax'))
seq_model.summary()

### Functional API

In the Functional API, you explicitly create the pipeline we discussed before by performing subsequent applications of all the functions, __starting from a placeholder defining the input tensor__.

In [ ]:
inputs = K.Input(shape=(28, 28)) # Implicitly, the shape will be (None, 28, 28)
flattened = K.layers.Flatten()(inputs)
rescaled = K.layers.Rescaling(scale=1/255.)(flattened)
outputs = K.layers.Dense(units=10, activation='softmax')(rescaled)
fn_model = K.Model(inputs=inputs, outputs=outputs)
fn_model.summary()

## K.Model API

This is the best chance you have to exploit the synergy between Keras and TensorFlow. You can define your own custom model by inheriting from the class ``K.Model``.

In [ ]:
@K.saving.register_keras_serializable()
class SimpleLinearClassifier(K.Model):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.flatten = K.layers.Flatten()
        self.rescale = K.layers.Rescaling(scale=1./255)
        self.dense = K.layers.Dense(units=10, activation='softmax')

    def call(self, inputs):
        x = self.flatten(inputs)
        x = self.rescale(x)
        return self.dense(x)

cl_model = SimpleLinearClassifier()
cl_model.summary()

# Compiling the pipeline

As mentioned at the beginning, the greatest advantage of Keras over TensorFlow is that our processing pipeline is compiled on a static computational graph.

In [ ]:
#model = seq_model
#model = fn_model
model = cl_model

In [ ]:
model.compile(
    optimizer=K.optimizers.Adam(learning_rate=0.1),
    loss=K.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Model Training

Training a model with Keras is as simple as calling the `.fit` method.

The ``callbacks`` parameter allows to add a list of functionalities which are called at specific times of the training loop.

In [ ]:
early_stopping = K.callbacks.EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=5,
    min_delta=0.0005,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    x=train_X,
    y=train_y,
    epochs=100,
    batch_size=1000,
    validation_data=(eval_X, eval_y),
    callbacks=[early_stopping]
)

And we have the additional advantage of the training history, providing statistics of the training phase useful for observing the behaviour of the model when the process is over.

In [ ]:
list(history.history)

In [ ]:
results = pd.DataFrame({
    'epoch': history.epoch,
    'loss': history.history['loss'],
    'val_loss': history.history['val_loss']
})
ax = sns.lineplot(x='epoch', y='value', hue='variable', data=pd.melt(results, ['epoch']))
ax.set(xlabel='epoch', ylabel='loss value')

In [ ]:
results = pd.DataFrame({
    'epoch': history.epoch,
    'loss': history.history['accuracy'],
    'val_loss': history.history['val_accuracy']
})
ax = sns.lineplot(x='epoch', y='value', hue='variable', data=pd.melt(results, ['epoch']))
ax.set(xlabel='epoch', ylabel='accuracy value')

Keras models expose the method `.evaluate`, which returns the results of the metrics on a given dataset.

In [ ]:
metrics = model.evaluate(test_X, test_y)
metrics # loss, accuracy

Instead, `.predict` returns the predictions of the model

In [ ]:
predictions = model.predict(test_X)
predictions

## Model serialization

A Keras Model can be easily `serialized` and `deserialized` as follows.

**Note:** custom `K.Model` subclasses must be decorated with `@K.saving.register_keras_serializable()` so that Keras can locate the class when loading the model back. Without it, `load_model` fails with a `TypeError`. The Sequential and Functional APIs do not need this because they are built from standard Keras layers that are already registered.

In [ ]:
model.save('my_model.keras', overwrite=True)

In [ ]:
loaded_model = K.models.load_model('my_model.keras')

There are several other options to serialize a model. It can be saved in H5 format, you can save only the parameters, only the model architecture etc..
You can find further details at the following webpage: https://www.tensorflow.org/guide/keras/save_and_serialize

In [ ]:
loaded_model.summary()

---
## Your Turn! — MNIST CNN

Implement and train a **Convolutional Neural Network** to perform image classification on the MNIST dataset.

In [ ]:
from keras.datasets import mnist
from sklearn.model_selection import train_test_split

(mnist_train_X, mnist_train_y), (mnist_test_X, mnist_test_y) = mnist.load_data(path='ds')

# TODO: Split mnist_train into train and validation

# TODO: Rescale and reshape the data for a CNN (add channel dimension)

In [ ]:
# TODO: Define your CNN model

mnist_model = None

In [ ]:
# TODO: Compile the model

# TODO: Train the model with early stopping

# TODO: Evaluate on the test set (accuracy!)

# TODO: Plot training and validation loss